---

## **Part 1: LangChain Framework – Core Architecture (Agentic Use Case Perspective)**

### **The Problem LangChain Solves**
LLMs (like GPT-4) are great at generating text, but:
- They can’t access tools like search, calculator, or your company data.
- They forget everything between messages unless you give them memory.
- They can't reason step-by-step or make decisions dynamically.

**LangChain solves this by acting as a bridge.**

---

### **LangChain Key Components (Agentic Perspective)**

| Component | What It Does | Real Example |
|----------|--------------|--------------|
| **LLM** | Connects to GPT-4, Claude, Mistral, etc. | "Write an email to HR" |
| **PromptTemplate** | Creates reusable prompt structures | "Write a tweet about {topic}" |
| **Tool** | External actions (Search, API call, DB) | Weather lookup, PDF reader |
| **Agent** | Decides *when* and *how* to use tools | "Hmm… to answer this, I need weather + location" |
| **Memory** | Stores past conversations | Multi-turn chatbots |
| **Retriever** | Retrieves documents from a knowledge base | Vector search in RAG |
| **Chain** | Links LLM calls together in steps | Extract from PDF → Generate summary → Send email |

---

### **Visual Architecture: Simple Agentic System**

```
[User Query]
     |
     v
[LangChain Agent] --- decides --->
     |                           |
     |                          [Tool 1: Weather API]
     |                          [Tool 2: Flight Search]
     |                          [Tool 3: Calculator]
     |
[Final Answer to User]
```


---



---

## **Problem Statement**

In many real-world scenarios, users seek contextual information about the current date and time as part of their natural language queries. For example:

* *“What is today’s date?”*
* *“Can you tell me the current time before I schedule a meeting?”*

While large language models (LLMs) can generate approximate or static answers, they are **not inherently aware of real-time system information**. As a result, they may provide outdated or fabricated responses when asked about the current date or time.

To ensure accuracy, an intelligent assistant must:

1. **Understand natural language queries referring to temporal context.**
2. **Automatically invoke tools that fetch real-time system date and time.**
3. **Integrate tool outputs into a conversational, human-friendly response.**

This functionality is crucial in **productivity tools, scheduling assistants, personal reminders, and business applications**, where real-time awareness significantly enhances user experience and reliability.

---

## **Objective**

Design and implement a **LangChain-powered agent** that integrates real-time utilities for date and time. The agent should:

* **Detect** when a user query involves current date or time.
* **Automatically call the appropriate tool** (`get_current_date` or `get_current_time`).
* **Leverage LangChain’s tool-calling framework** to handle reasoning and orchestration.
* **Maintain a reasoning trace** using `agent_scratchpad` for clarity.
* **Provide responses** in a conversational and context-aware format, ensuring outputs are both precise and user-friendly.

---



In [1]:
# !pip install -U \
# langchain==0.3.27 \
# langchain-core==0.3.79 \
# langchain-community==0.3.31 \
# langchain-openai==0.3.35 \
# langchain-text-splitters==0.3.9 \
# langchainhub \
# faiss-cpu \
# sentence-transformers -q

In [2]:
# Agent components:
# 1. Brain: LLM (e.g., GPT-4, Claude 3)
# 2. Prompt Template: Guides the LLM's responses.
# 3. Tools: Functions the agent can call to perform tasks.
# 4. Memory: Stores context from previous interactions.

In [3]:
# Langchain Simple Agent

# Import required libraries

# For calling the Openai API
from langchain_openai import ChatOpenAI # LLM BRAIN - ENGINE

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # MESSAGES - CHASSIS
# ChatPromptTemplate: It is used to create a prompt template for chat models. This helps in structuring the input for the model.
# MessagesPlaceholder: It is used to indicate where the messages should be placed in the prompt template. This allows for dynamic insertion of messages during the conversation.

from langchain.tools import tool # HANDS - DASHBOARD
# tool: It is used to define a tool that can be called by the agent. This allows the agent to perform specific actions or retrieve information.

from langchain.agents import create_tool_calling_agent, AgentExecutor
# create_tool_calling_agent: It is used to create an agent that can call tools. This allows the agent to use the defined tools to perform tasks.
# AgentExecutor: It is used to execute the agent. This allows the agent to process inputs

In [4]:
# Load the environment variables from the .env file
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

True

In [5]:
llm = ChatOpenAI(model="gpt-4o")
# https://python.langchain.com/docs/integrations/chat/openai/

In [6]:
# Step 5: Create prompt template (IMPORTANT: include input & agent_scratchpad)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a smart assistant. You can use tools whenever necessary to answer user questions."),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")  # Required
    # MessagesPlaceholder - A special placeholder used by LangChain agents to store: a. Intermediate reasoning steps, b. Tool calls, c. Tool outputs - ReAct Style (Thoughts, Actions, and Observations)
])

In [22]:
from datetime import datetime

now = datetime.now()
print("Date:", now.strftime("%Y-%m-%d"))
print("Time:", now.strftime("%H:%M:%S"))

Date: 2026-09-24
Time: 10:50:20


In [23]:
# Install pendulum package for timezone handling
# !pip install pendulum -q

import pendulum

now = pendulum.now("Asia/Kolkata")
print("Date:", now.format("YYYY-MM-DD"))
print("Time:", now.format("HH:mm:ss"))

Date: 2026-09-24
Time: 10:51:01


In [26]:
import requests
from datetime import datetime, timezone
import ntplib

# What is ntplib?
# ntplib is a Python library used to interact with Network Time Protocol (NTP) servers. It allows you to retrieve the current time from NTP servers, which is useful for applications that require accurate timekeeping.
# This function fetches the current UTC time from an internet source using ntplib and falls back to Google's server time if ntplib fails.

def get_internet_utc():
    try:
        client = ntplib.NTPClient()
        response = client.request("pool.ntp.org")
        print("Hitting WorldTimeAPI, got data:", response)
        return datetime.fromtimestamp(response.tx_time, timezone.utc)
    except Exception:
        r = requests.head("https://google.com")
        server_time = r.headers["Date"]
        print("Hitting Google, got server time:", server_time)
        return datetime.strptime(server_time, "%a, %d %b %Y %H:%M:%S %Z").replace(tzinfo=timezone.utc)

utc_now = get_internet_utc()
print("Reliable UTC time:", utc_now)


Hitting WorldTimeAPI, got data: <ntplib.NTPStats object at 0x17648d410>
Reliable UTC time: 2026-09-24 05:24:27.647851+00:00


In [27]:
# Tool Definitions
import ntplib
import requests
import psutil
import platform

@tool
def get_current_utc_date_and_time():
    """Returns the current date and time in UTC (Coordinated Universal Time) in the format YYYY-MM-DD HH:mm:ss"""
    try:
        client = ntplib.NTPClient()
        response = client.request("pool.ntp.org")
        print("Hitting WorldTimeAPI, got data:", response)
        current_time_utc = datetime.fromtimestamp(response.tx_time, timezone.utc)
    except Exception:
        r = requests.head("https://google.com")
        server_time = r.headers["Date"]
        print("Hitting Google, got server time:", server_time)
        current_time_utc = datetime.strptime(server_time, "%a, %d %b %Y %H:%M:%S %Z").replace(tzinfo=timezone.utc)

    return f"The current date and time in UTC is: {current_time_utc.strftime('%Y-%m-%d %H:%M:%S')}"

tools = [get_current_utc_date_and_time]

In [28]:
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [29]:
# Wrap the agent in an AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True # Set verbose=True to see the agent's reasoning and tool calls in the output
)

In [10]:
response = agent_executor.invoke({"input": "Explain the process of photosynthesis in 5 lines."})
print(response['output'])
# print(response)

# How does the Agent work?
# 1. The user provides an input query to the agent.
# 2. The query "input" is passed to the agent's prompt template, which structures the input for the LLM.
# 3. This structured prompt is then sent to the LLM (e.g., GPT-4) for processing. LLM is good at understanding and generating human-like text based on the input it receives.
# 4. The LLM processes the input and looks for any tool calls if available. If the LLM determines that a tool is available, then it will call the tool with the appropriate parameters and get back the output from the tool. However, if the LLM determines that no tool is available, then it will see if it can answer the question itself based on its training data and the context provided in the prompt. If it can answer the question, it will generate a response directly. If it cannot answer the question, it will generate a response indicating that it cannot answer the question.
# 5. If the tool is called, the output from the tool is then passed back to the LLM as part of the agent's scratchpad (intermediate reasoning steps). The LLM can then use this information to generate a final response to the user's query.
# 6. The final response is then returned to the user.

# Ques -> AGENT(llm+tools+prompt) -> Prompt -> llm(brain) -> check for tools -> No tools -> llm(brain) -> Can I answer? -> Yes/No -> If No -> I cannot answer. If Yes -> Generate response -> Return response to user



> Entering new AgentExecutor chain...
Photosynthesis is the process by which green plants, algae, and some bacteria convert light energy into chemical energy. It occurs primarily in the chloroplasts of plant cells where chlorophyll captures sunlight. This energy is used to convert carbon dioxide and water into glucose and oxygen. The general equation for photosynthesis is: 6CO2 + 6H2O + light energy → C6H12O6 + 6O2. It is crucial for producing oxygen and serving as the foundation of the food chain.

> Finished chain.
Photosynthesis is the process by which green plants, algae, and some bacteria convert light energy into chemical energy. It occurs primarily in the chloroplasts of plant cells where chlorophyll captures sunlight. This energy is used to convert carbon dioxide and water into glucose and oxygen. The general equation for photosynthesis is: 6CO2 + 6H2O + light energy → C6H12O6 + 6O2. It is crucial for producing oxygen and serving as the foundation of the food chain.


In [30]:
response = agent_executor.invoke({"input": "Explain the process of photosynthesis in 5 bullet points."})
print(response['output'])




> Entering new AgentExecutor chain...
- **Light Absorption**: Photosynthesis begins when chlorophyll in the plant's chloroplasts absorbs sunlight. This energy is used to excite electrons, initiating the process.

- **Water Splitting (Photolysis)**: The absorbed light energy splits water molecules into oxygen, protons, and electrons. This occurs in the thylakoid membranes of the chloroplasts.

- **Energy Conversion (Light Reactions)**: The excited electrons move through the electron transport chain, generating ATP (adenosine triphosphate) and NADPH (nicotinamide adenine dinucleotide phosphate) through a series of redox reactions.

- **Carbon Fixation (Calvin Cycle)**: The ATP and NADPH produced in the light reactions are used in the stroma of the chloroplasts to convert carbon dioxide (CO2) into glucose during the Calvin Cycle.

- **Glucose Formation**: The carbon dioxide, ATP, and NADPH are utilized to create glucose, which serves as an energy source for the plant, while oxygen is re

In [12]:
response = agent_executor.invoke({"input": "What is the capital of India?"})
print(response['output'])




> Entering new AgentExecutor chain...
The capital of India is New Delhi.

> Finished chain.
The capital of India is New Delhi.


In [13]:
response = agent_executor.invoke({"input": "What is the capital of New Delhi?"})
print(response['output'])




> Entering new AgentExecutor chain...
New Delhi itself is the capital of India. It is a city and a district where the central government of India is located.

> Finished chain.
New Delhi itself is the capital of India. It is a city and a district where the central government of India is located.


In [14]:
response = agent_executor.invoke({"input": "My name is Darshan and I want to become a data scientist. Please guide me with three successful career tips."})
print(response['output'])




> Entering new AgentExecutor chain...
Becoming a data scientist is an exciting journey, and here are three tips to help you succeed in this field:

1. **Continuous Learning and Skill Development:**
   - Data science is a rapidly evolving field, so it's crucial to continuously update your skills. Focus on mastering fundamental skills such as programming (Python, R), statistics, and data manipulation.
   - Stay current with the latest tools and technologies in data science, such as machine learning frameworks (e.g., TensorFlow, PyTorch) and data visualization tools (e.g., Tableau, Power BI).
   - Online courses, certifications, and workshops can be a great way to keep learning.

2. **Hands-on Practice and Projects:**
   - Apply your knowledge through real-world projects and problem-solving. This could be through internships, contributing to open-source projects, or personal projects.
   - Build a portfolio of your work to showcase your ability to analyze data and generate insights. Web

In [15]:
response = agent_executor.invoke({"input": "What is the capital of New York?"})
print(response['output'])




> Entering new AgentExecutor chain...
The capital of New York is Albany.

> Finished chain.
The capital of New York is Albany.


In [16]:
response = agent_executor.invoke({"input": "Do do you recollect my name?"})
print(response['output'])




> Entering new AgentExecutor chain...
I'm sorry, but I don't seem to have access to your name or any previous interactions. Could you please remind me?

> Finished chain.
I'm sorry, but I don't seem to have access to your name or any previous interactions. Could you please remind me?


In [17]:
response = agent_executor.invoke({"input": "My name is Darshan and I want to become a data scientist. Please guide me with three successful career tips. At the end, give me a personalised tip, mentioning my name."})
print(response['output'])




> Entering new AgentExecutor chain...
Of course, Darshan! Here are three successful career tips for becoming a data scientist:

1. **Master the Fundamentals**: Ensure you have a strong grasp of statistics, mathematics, and programming. Proficiency in languages like Python and R, as well as knowledge of machine learning algorithms and data manipulation libraries (such as pandas, NumPy, or Scikit-learn), is crucial.

2. **Build a Portfolio**: Work on real-world projects and datasets to build a strong portfolio. Participate in competitions on platforms like Kaggle, collaborate on open-source projects, and showcase your work on GitHub or a personal website to demonstrate your skills and problem-solving ability.

3. **Stay Current and Network**: Data science is a rapidly evolving field. Keep up with the latest trends, tools, and research by following relevant blogs, attending webinars, and participating in workshops or conferences. Networking with other data science professionals can also

In [18]:
response = agent_executor.invoke({"input": "Do do you recollect my name?"})
print(response['output'])




> Entering new AgentExecutor chain...
I'm sorry, but I can't recall personal information or previous interactions. Could you please share your name with me again?

> Finished chain.
I'm sorry, but I can't recall personal information or previous interactions. Could you please share your name with me again?


In [31]:
response = agent_executor.invoke({"input": "What is today's date?"}) # ambiguous question - no timezone mentioned
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Hitting WorldTimeAPI, got data: <ntplib.NTPStats object at 0x17888ccd0>
The current date and time in UTC is: 2026-09-24 05:27:54Today's date is September 24, 2026.

> Finished chain.
Today's date is September 24, 2026.


In [32]:
response = agent_executor.invoke({"input": "What is today's date?"}) # ambiguous question - no timezone mentioned
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Hitting Google, got server time: Thu, 24 Sep 2026 05:30:56 GMT
The current date and time in UTC is: 2026-09-24 05:30:56Today's date, in UTC, is September 24, 2026.

> Finished chain.
Today's date, in UTC, is September 24, 2026.


In [33]:
response = agent_executor.invoke({"input": "What is today's date and time?"}) # ambiguous question - no timezone mentioned
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Hitting WorldTimeAPI, got data: <ntplib.NTPStats object at 0x17888dad0>
The current date and time in UTC is: 2026-09-24 05:31:00Today's date and time in UTC is 2026-09-24 05:31:00.

> Finished chain.
Today's date and time in UTC is 2026-09-24 05:31:00.


In [34]:
response = agent_executor.invoke({"input": "What is today's date and current time in India?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Hitting WorldTimeAPI, got data: <ntplib.NTPStats object at 0x1788b1b90>
The current date and time in UTC is: 2026-09-24 05:31:31To find the current date and time in India, we need to convert from UTC to Indian Standard Time (IST), which is UTC+5:30. 

Given the current UTC time is 2026-09-24 05:31:31, the current time in India is 2026-09-24 11:01:31.

> Finished chain.
To find the current date and time in India, we need to convert from UTC to Indian Standard Time (IST), which is UTC+5:30. 

Given the current UTC time is 2026-09-24 05:31:31, the current time in India is 2026-09-24 11:01:31.


In [35]:
response = agent_executor.invoke({"input": "Is it 24th September 2026 in India today?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Hitting WorldTimeAPI, got data: <ntplib.NTPStats object at 0x17881c090>
The current date and time in UTC is: 2026-09-24 05:32:46The current date and time in UTC is September 24, 2026, 05:32:46.

India is 5 hours and 30 minutes ahead of UTC, so the local time in India is September 24, 2026, 11:02:46 AM. Therefore, it is indeed September 24, 2026 in India today.

> Finished chain.
The current date and time in UTC is September 24, 2026, 05:32:46.

India is 5 hours and 30 minutes ahead of UTC, so the local time in India is September 24, 2026, 11:02:46 AM. Therefore, it is indeed September 24, 2026 in India today.


In [36]:
response = agent_executor.invoke({"input": "Is it Thursday today in India?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Hitting WorldTimeAPI, got data: <ntplib.NTPStats object at 0x17649c690>
The current date and time in UTC is: 2026-09-24 05:33:38To determine if it's Thursday in India, we need to convert the current UTC time to India Standard Time (IST), which is UTC+5:30.

The current UTC time is 2026-09-24 05:33:38. Adding 5 hours and 30 minutes gives us 2026-09-24 11:03:38 IST.

Since the date is September 24, 2026, and a simple date check can tell us the day:

September 24, 2026, is indeed a Thursday.

Therefore, today is Thursday in India.

> Finished chain.
To determine if it's Thursday in India, we need to convert the current UTC time to India Standard Time (IST), which is UTC+5:30.

The current UTC time is 2026-09-24 05:33:38. Adding 5 hours and 30 minutes gives us 2026-09-24 11:03:38 IST.

Since the date is September 24, 2026, and a simple date check can tell us the day:

September 24, 2026, is indee

In [37]:
response = agent_executor.invoke({"input": "What is today's date and time in Pakistan?"})
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Hitting WorldTimeAPI, got data: <ntplib.NTPStats object at 0x1788b76d0>
The current date and time in UTC is: 2026-09-24 05:34:09The current date and time in UTC is 2026-09-24 05:34:09. Pakistan Standard Time is UTC+5 hours. Let's calculate the current date and time in Pakistan:

UTC time: 2026-09-24 05:34:09  
Adding 5 hours: 2026-09-24 10:34:09  

So, the current date and time in Pakistan is 2026-09-24 10:34:09.

> Finished chain.
The current date and time in UTC is 2026-09-24 05:34:09. Pakistan Standard Time is UTC+5 hours. Let's calculate the current date and time in Pakistan:

UTC time: 2026-09-24 05:34:09  
Adding 5 hours: 2026-09-24 10:34:09  

So, the current date and time in Pakistan is 2026-09-24 10:34:09.


In [38]:
response = agent_executor.invoke({"input": "What is today's date and time in NYC, USA?"})
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Hitting WorldTimeAPI, got data: <ntplib.NTPStats object at 0x176421210>
The current date and time in UTC is: 2026-09-24 05:34:26The current UTC date and time is 2026-09-24 05:34:26. New York City, USA, is in the Eastern Time Zone, which is 4 hours behind UTC during Daylight Saving Time (EDT). Therefore, the current date and time in NYC is 2026-09-24 01:34:26.

> Finished chain.
The current UTC date and time is 2026-09-24 05:34:26. New York City, USA, is in the Eastern Time Zone, which is 4 hours behind UTC during Daylight Saving Time (EDT). Therefore, the current date and time in NYC is 2026-09-24 01:34:26.


In [39]:
response = agent_executor.invoke({"input": "What is today's date and time in Winnipeg, Canada?"})
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Hitting WorldTimeAPI, got data: <ntplib.NTPStats object at 0x1788b6050>
The current date and time in UTC is: 2026-09-24 05:34:48The current date and time in UTC is 2026-09-24 05:34:48. Winnipeg, Canada is in the Central Time Zone, which is UTC-5 during standard time and UTC-6 when daylight saving time is in effect.

Since it's currently September, daylight saving time is typically observed. Therefore, the current time in Winnipeg would be UTC-5.

Let's calculate the Winnipeg time:

- UTC date and time: 2026-09-24 05:34:48
- Subtract 5 hours for Winnipeg time (during daylight saving time):

So, the corresponding date and time in Winnipeg is 2026-09-24 00:34:48.

> Finished chain.
The current date and time in UTC is 2026-09-24 05:34:48. Winnipeg, Canada is in the Central Time Zone, which is UTC-5 during standard time and UTC-6 when daylight saving time is in effect.

Since it's currently Septem

In [40]:
response = agent_executor.invoke({"input": "What is today's date and time in Honalulu, USA?"})
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Hitting WorldTimeAPI, got data: <ntplib.NTPStats object at 0x1788b2590>
The current date and time in UTC is: 2026-09-24 05:35:19To determine the current date and time in Honolulu, USA, we need to convert from UTC to Honolulu's local time, which observes Hawaii Standard Time (HST) and is UTC-10. 

Given the UTC time is 2026-09-24 05:35:19, the time in Honolulu is 10 hours behind:

- **Honolulu Time**: 2026-09-23 19:35:19 (7:35 PM) HST

> Finished chain.
To determine the current date and time in Honolulu, USA, we need to convert from UTC to Honolulu's local time, which observes Hawaii Standard Time (HST) and is UTC-10. 

Given the UTC time is 2026-09-24 05:35:19, the time in Honolulu is 10 hours behind:

- **Honolulu Time**: 2026-09-23 19:35:19 (7:35 PM) HST


In [41]:
response = agent_executor.invoke({"input": "How many days have passed since the latest India's Independence Day?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Hitting Google, got server time: Thu, 24 Sep 2026 05:36:17 GMT
The current date and time in UTC is: 2026-09-24 05:36:17India's Independence Day is celebrated on August 15th. The current UTC date is September 24, 2026.

To find out how many days have passed since the latest India's Independence Day, we calculate the difference between these two dates:

- Current Date: September 24, 2026
- Independence Day: August 15, 2026

1. Calculate the difference in days between August 15 and September 24, 2026.

From August 15 to September 15 is 31 days, and from September 15 to September 24 is 9 days. 

Therefore, 31 + 9 = 40 days have passed since the latest India's Independence Day.

> Finished chain.
India's Independence Day is celebrated on August 15th. The current UTC date is September 24, 2026.

To find out how many days have passed since the latest India's Independence Day, we calculate the differ

In [42]:
response = agent_executor.invoke({"input": "How many days have passed since the latest India's Independence Day? Think step-by-step."})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Hitting Google, got server time: Thu, 24 Sep 2026 05:36:52 GMT
The current date and time in UTC is: 2026-09-24 05:36:52Let's calculate how many days have passed since India's latest Independence Day, which is celebrated on August 15th every year.

1. **Identify the Current Date:** The current date is September 24, 2026.

2. **Calculate the Difference between the Current Date and Latest Independence Day:**

   - The latest Independence Day was on August 15, 2026.
   - From August 15 to September 24 is a span within the same year 2026.

3. **Days Calculation:**
   - August 15 to August 31 = 16 days
   - September 1 to September 24 = 24 days

Adding these together: \(16 + 24 = 40\) days.

Therefore, 40 days have passed since India's latest Independence Day as of the current date, September 24, 2026.

> Finished chain.
Let's calculate how many days have passed since India's latest Independence Da